# AI Meeting Assistant — Stage 2 (Domain-aware transcript refinement, LLM #1)

**Runtime → Change runtime type → T4 GPU.** Run Stage 1 first; this notebook takes its `stage1_record.json`.

What this stage does: the raw transcript goes in, a **list of small, validated edits** comes out, and the refined
transcript is produced by applying only the edits that pass deterministic checks. The LLM never rewrites text.

| Step | What | Cell |
|---|---|---|
| Load | `stage1_record.json` → segments, low-confidence words | 4 |
| Protect | find numbers, negations, modals, names that must never change | 5 |
| Chunk | 40 segments per call, 5 overlap, stable IDs | 6 |
| Prompt | Appendix-A refiner prompt + few-shot examples | 7 |
| Validate | 8 deterministic checks per proposed edit | 8–9 |
| LLM | Qwen3-8B, 4-bit, thinking off, temperature 0 | 10–11 |
| Run / inspect / save | refined transcript, edit log, diff, metrics | 12–16 |

Declared refiner model for the submission: **Qwen/Qwen3-8B (Apache-2.0), 4-bit NF4 via bitsandbytes**.
Swap to `meta-llama/Llama-3.1-8B-Instruct` by changing one line in Cell 3 once your HF access is approved.

## Cell 1 — Runtime check

In [ ]:
import subprocess, sys
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    gpu = ""
print("GPU :", gpu or "NONE -> Runtime > Change runtime type > T4 GPU")
print("python:", sys.version.split()[0])

GPU : Tesla T4, 15360 MiB, 0 MiB
python: 3.13.15


In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    MODELS_DIR = Path("/content/drive/MyDrive/meeting-assistant/models")
except Exception as e:
    print("Drive not mounted ->", type(e).__name__, "| using local cache (lost on runtime reset)")
    MODELS_DIR = Path("/content/models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print("Model cache:", MODELS_DIR)
for p in sorted(MODELS_DIR.glob("*")):
    size = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e9
    print(f"  cached: {p.name:<40} {size:5.1f} GB")

## Cell 2 — Install

In [ ]:
!pip install -q "transformers>=4.51" accelerate bitsandbytes rapidfuzz jellyfish

import transformers, torch
print("transformers", transformers.__version__, "| torch", torch.__version__, "| cuda:", torch.cuda.is_available())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.4/360.4 kB 20.0 MB/s eta 0:00:00
transformers 5.18.0 | torch 2.11.0+cu130 | cuda: True


## Cell 3 — Configuration (becomes `config.py` → refiner section)

In [ ]:
from dataclasses import dataclass, field
from pathlib import Path

@dataclass
class RefineConfig:
    # ---- model ----
    model_id: str      = "Qwen/Qwen3-8B"            # or "meta-llama/Llama-3.1-8B-Instruct" (gated)
    load_in_4bit: bool = True
    max_new_tokens: int = 600
    # ---- chunking ----
    chunk_segments: int = 40
    chunk_overlap: int  = 5
    # ---- validators ----
    max_edits_per_chunk: int = 8
    max_original_words: int  = 4          # an edit may touch at most this many words
    max_word_delta: int      = 2          # replacement may differ in length by at most this many words
    min_confidence: float    = 0.6        # model's own confidence; below this -> rejected
    fuzzy_min_ratio: int     = 55         # rapidfuzz ratio (0-100) OR metaphone match OR glossary hit
    # ---- context ----
    meeting_context: str = "Technical team meeting (software / engineering)."
    glossary: list = field(default_factory=list)    # user-supplied terms; also auto-filled from Stage 1 hints

CFG = RefineConfig()
WORK_DIR = Path("/content/meeting-assistant"); OUT_DIR = WORK_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CFG

RefineConfig(model_id='Qwen/Qwen3-8B', load_in_4bit=True, max_new_tokens=600, chunk_segments=40, chunk_overlap=5, max_edits_per_chunk=8, max_original_words=4, max_word_delta=2, min_confidence=0.6, fuzzy_min_ratio=55, meeting_context='Technical team meeting (software / engineering).', glossary=[])

## Cell 4 — Load the Stage 1 output

Upload `stage1_record.json` (from the Stage 1 notebook's Cell 15) or point `STAGE1_PATH` at it on Drive.
Also accepts the flat `raw_transcript.json` shape produced by `pipeline_stt.py`.

In [ ]:
import json
from google.colab import files

uploaded = files.upload()
STAGE1_PATH = str(WORK_DIR / next(iter(uploaded)))
Path(STAGE1_PATH).write_bytes(next(iter(uploaded.values())))
# STAGE1_PATH = "/content/drive/MyDrive/meeting-assistant/outputs/<run>/stage1_record.json"

def load_stage1(path: str) -> dict:
    d = json.loads(Path(path).read_text(encoding="utf-8"))
    if "raw_transcript" in d and isinstance(d["raw_transcript"], dict):          # Stage 1 notebook shape
        segs = d["raw_transcript"]["segments"]
        low  = d.get("low_confidence_words", [])
        meta = d.get("meta", {})
    elif "segments" in d:                                                        # pipeline_stt.py shape
        segs = [{"id": s["id"], "start": s["start"], "end": s["end"], "text": s["text"]} for s in d["segments"]]
        low  = [{"segment_id": s["id"], "word": w} for s in d["segments"] for w in s.get("low_conf_words", [])]
        meta = {"models": {"stt": d.get("info", {}).get("model")}, "duration_s": d.get("info", {}).get("duration")}
    else:
        raise ValueError("Unrecognised Stage 1 file: expected 'raw_transcript' or 'segments'.")
    for s in segs:
        s["text"] = s["text"].strip()
    return {"segments": segs, "low_confidence_words": low, "meta": meta, "record": d}

S1 = load_stage1(STAGE1_PATH)
RAW_SEGMENTS = S1["segments"]
print(f"{len(RAW_SEGMENTS)} segments | {sum(len(s['text'].split()) for s in RAW_SEGMENTS)} words | "
      f"{len(S1['low_confidence_words'])} low-confidence words | STT = {S1['meta'].get('models', {}).get('stt')}")
print("\n".join(f"[{s['id']:03d}] {s['text']}" for s in RAW_SEGMENTS[:6]))

Saving stage1_record.json to stage1_record (4).json
3 segments | 63 words | 1 low-confidence words | STT = faster-whisper 1.2.1 / whisper-large-v3
[001] We evaluated the pie torch model with low ra adaptations. The word error rate was zero point zero eight, not zero point eight.
[002] Do not increase the batch size beyond 64 because the cuda memory will overflow.
[003] We also observed that the embedding vectors from bert did not converge when using the atom optimizer with weight decay. Let's switch to adam w instead.


## Cell 5 — Protected tokens (the things an edit must never touch)

Digits, number words, negations, modal/commitment words, and capitalised names. Computed from the raw text,
stored in the record, and used by the validators. Names in the glossary are the one exception (the user vouched
for the spelling).

In [ ]:
import re
from collections import Counter

NUMBER_WORDS = set('''zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen fifteen
sixteen seventeen eighteen nineteen twenty thirty forty fifty sixty seventy eighty ninety hundred thousand million
billion first second third fourth fifth sixth seventh eighth ninth tenth half quarter dozen twice'''.split())
NEGATIONS = set('''not no never none nothing neither nor cannot without don't doesn't didn't won't wouldn't can't
couldn't shouldn't isn't aren't wasn't weren't hasn't haven't hadn't mustn't'''.split())
MODALS = set("will shall should must may might can could would agreed agree decided decide commit".split())
PROTECTED_WORDS = NUMBER_WORDS | NEGATIONS | MODALS

def tokens(text: str) -> list[str]:
    return re.findall(r"[A-Za-z0-9']+(?:/[A-Za-z0-9']+)?", text)

def protected_in(text: str) -> list[str]:
    "Lower-cased protected tokens (digits, number words, negations, modals) in order of appearance."
    out = []
    for t in tokens(text):
        tl = t.lower()
        if re.fullmatch(r"\d[\d,.:%]*", t) or tl in PROTECTED_WORDS or tl.endswith("n't"):
            out.append(tl)
    return out

def capitalised_names(text: str, glossary: list[str]) -> list[str]:
    "Capitalised tokens that are not sentence-initial and not in the glossary -> treated as names."
    gl = {g.lower() for g in glossary}
    toks = tokens(text); names = []
    for i, t in enumerate(toks):
        if i > 0 and t[0].isupper() and t.lower() not in gl and not t.isupper() and len(t) > 1:
            names.append(t)
    return names

SENTENCE_STARTERS = set("the a an and but so or if then first second third next okay ok yes no we you i it this that "
                        "these those there here someone everyone nobody anyone also finally lastly today now please "
                        "thanks thank right well".split()) | {"let's", "that's", "it's", "we're", "they're"}

def global_names(segments: list, glossary: list[str]) -> set[str]:
    # Name-like tokens across the whole transcript: capitalised mid-sentence anywhere, OR capitalised and
    # repeated (>=2 times, any position, so sentence-initial names like "Priya says..." are still caught).
    gl = {g.lower() for g in glossary}; names, counts = set(), Counter()
    for s in segments:
        names.update(capitalised_names(s["text"], glossary))
        for t in tokens(s["text"]):
            if t[0].isupper() and not t.isupper() and len(t) > 1 and t.lower() not in gl | SENTENCE_STARTERS | PROTECTED_WORDS:
                counts[t] += 1
    names.update(t for t, n in counts.items() if n >= 2)
    return names

def build_protected_index(segments: list, glossary: list[str]) -> dict:
    return {s["id"]: {"protected": protected_in(s["text"]), "names": capitalised_names(s["text"], glossary)}
            for s in segments}

PROTECTED = build_protected_index(RAW_SEGMENTS, CFG.glossary)
NAMES = global_names(RAW_SEGMENTS, CFG.glossary)
print("name-like tokens (never respelled unless in glossary):", sorted(NAMES))
n_prot = sum(len(v["protected"]) for v in PROTECTED.values()); n_names = sum(len(v["names"]) for v in PROTECTED.values())
print(f"protected tokens: {n_prot} | name-like tokens: {n_names}")
for sid, v in list(PROTECTED.items())[:5]:
    if v["protected"] or v["names"]:
        print(f"  seg {sid:03d}: protected={v['protected']} names={v['names']}")

name-like tokens (never respelled unless in glossary): ["Let's", 'The']
protected tokens: 10 | name-like tokens: 2
  seg 001: protected=['zero', 'zero', 'eight', 'not', 'zero', 'eight'] names=['The']
  seg 002: protected=['not', '64', 'will'] names=[]
  seg 003: protected=['not'] names=["Let's"]


## Cell 6 — Chunker + suspect spans + auto-glossary

Overlapping chunks keep stable segment IDs; edits in the overlap are de-duplicated later. The low-confidence
words from Stage 1 become *suspect spans* the model is told to look at first. A cheap auto-glossary (capitalised
tokens and acronyms that appear ≥2 times with consistent spelling) is added to the user glossary.

In [ ]:
from collections import Counter

def make_chunks(segments: list, size: int, overlap: int) -> list[list[dict]]:
    chunks, i = [], 0
    while i < len(segments):
        chunks.append(segments[i:i + size])
        if i + size >= len(segments):
            break
        i += size - overlap
    return chunks

def auto_glossary(segments: list, min_count: int = 2) -> list[str]:
    c = Counter()
    for s in segments:
        for i, t in enumerate(tokens(s["text"])):
            if (t.isupper() and 2 <= len(t) <= 8) or (i > 0 and t[0].isupper() and len(t) > 2):
                c[t] += 1
    return sorted(t for t, n in c.items() if n >= min_count and t.lower() not in PROTECTED_WORDS)

GLOSSARY = sorted(set(CFG.glossary) | set(auto_glossary(RAW_SEGMENTS)))
PROTECTED = build_protected_index(RAW_SEGMENTS, GLOSSARY)          # names in glossary are now editable
NAMES = global_names(RAW_SEGMENTS, GLOSSARY)
SUSPECT = {}
for w in S1["low_confidence_words"]:
    SUSPECT.setdefault(w["segment_id"], []).append(w["word"].strip(" .,?!"))

CHUNKS = make_chunks(RAW_SEGMENTS, CFG.chunk_segments, CFG.chunk_overlap)
print(f"{len(CHUNKS)} chunk(s) | glossary ({len(GLOSSARY)}): {GLOSSARY[:20]}")
print(f"suspect spans in {len(SUSPECT)} segments, e.g. {dict(list(SUSPECT.items())[:3])}")

1 chunk(s) | glossary (0): []
suspect spans in 1 segments, e.g. {2: ['sheek']}


## Cell 7 — Prompts (copy both into `prompts/refine_system.txt` for the repo)

The system prompt is the playbook's Appendix A, tightened. The user message carries the chunk, glossary and
suspect spans. The model must return only `{"edits": [...]}`.

In [ ]:
SYSTEM_PROMPT = '''You are an ASR error corrector, not an editor. You receive part of a meeting transcript produced by speech-to-text, one segment per line as [ID] text.
Your ONLY job: find words or phrases that are clearly misrecognised domain-specific terms, acronyms, product/tool names, or technical vocabulary, and propose the intended term.

Rules:
- Propose an edit only if the replacement sounds similar to the original AND fits the meeting context.
- NEVER change numbers, dates, times, quantities, names of people, negations (not, n't, never, no, cannot, without), or words of commitment (will, won't, should, must, might, can, agreed, decided).
- NEVER fix grammar, remove filler words (um, uh, like, I mean), paraphrase, summarise, reorder, or merge segments.
- Do not edit a person's name unless the glossary lists the correct spelling.
- If unsure, do not edit. Returning an empty list is correct when nothing is clearly wrong.
- "original" must be copied exactly (character for character) from the given segment, and must be at most 4 words.
- Return ONLY JSON, no commentary: {"edits":[{"segment_id":int,"original":str,"replacement":str,"reason":str,"confidence":float}]}

Examples:
[12] the post grass database crashed twice      -> {"segment_id":12,"original":"post grass","replacement":"PostgreSQL","reason":"phonetic match, database name","confidence":0.9}
[20] we need to push it to rod by friday        -> {"segment_id":20,"original":"rod","replacement":"prod","reason":"deployment target","confidence":0.85}
[31] the see eye see dee pipeline is red        -> {"segment_id":31,"original":"see eye see dee","replacement":"CI/CD","reason":"spelled-out acronym","confidence":0.9}
[44] we will not migrate to aws this quarter    -> no edit (negation and commitment; "aws" is already right)
[50] fifteen minutes, down from fifty           -> no edit (numbers are never changed)'''

def build_user_message(chunk: list[dict], glossary: list[str], suspect: dict, context: str) -> str:
    lines = [f"Meeting context: {context}"]
    if glossary:
        lines.append("Glossary (correct spellings of terms known to appear): " + ", ".join(glossary[:60]))
    sus = [f"[{s['id']}] {', '.join(suspect[s['id']])}" for s in chunk if s["id"] in suspect]
    if sus:
        lines.append("Low-confidence words from the speech recogniser (check these first): " + "; ".join(sus))
    lines.append("\nTranscript segments:")
    lines += [f"[{s['id']}] {s['text']}" for s in chunk]
    lines.append('\nReturn ONLY the JSON object {"edits":[...]}.')
    return "\n".join(lines)

print(build_user_message(CHUNKS[0][:5], GLOSSARY, SUSPECT, CFG.meeting_context)[:1200])

Meeting context: Technical team meeting (software / engineering).
Low-confidence words from the speech recogniser (check these first): [2] sheek

Transcript segments:
[1] We evaluated the pie torch model with low ra adaptations. The word error rate was zero point zero eight, not zero point eight.
[2] Do not increase the batch size beyond 64 because the cuda memory will overflow.
[3] We also observed that the embedding vectors from bert did not converge when using the atom optimizer with weight decay. Let's switch to adam w instead.

Return ONLY the JSON object {"edits":[...]}.


## Cell 8 — Validators (the part that earns the 20 points)

Every proposed edit must pass all checks. Rejections are logged with a reason — they go into the tech note.

In [ ]:
from rapidfuzz import fuzz
import jellyfish

# spelled-out letters -> letter, so "see eye see dee" ~ "CI/CD" and "A P I" ~ "API"
LETTER_WORDS = {"a":"a","ay":"a","bee":"b","be":"b","see":"c","sea":"c","cee":"c","dee":"d","e":"e","ee":"e","ef":"f",
    "gee":"g","aitch":"h","eye":"i","i":"i","jay":"j","kay":"k","el":"l","em":"m","en":"n","oh":"o","o":"o","pee":"p",
    "queue":"q","cue":"q","are":"r","es":"s","ess":"s","tee":"t","tea":"t","you":"u","u":"u","vee":"v","double":"w",
    "ex":"x","why":"y","zee":"z","zed":"z"}

def _alnum(s: str) -> str:
    return re.sub(r"[^a-z0-9]", "", s.lower())

def _spelled(s: str) -> str | None:
    ws = [w.lower() for w in tokens(s)]
    if ws and all(w in LETTER_WORDS or len(w) == 1 for w in ws):
        return "".join(LETTER_WORDS.get(w, w) for w in ws)
    return None

def phonetically_plausible(original: str, replacement: str, glossary: list[str], cfg: RefineConfig) -> tuple[bool, str]:
    o, r = _alnum(original), _alnum(replacement)
    if replacement.strip().lower() in {g.lower() for g in glossary}:
        return True, "glossary"
    if _spelled(original) == r:
        return True, "spelled-letters"
    if jellyfish.metaphone(o) == jellyfish.metaphone(r) and o and r:
        return True, "metaphone"
    ratio = fuzz.ratio(o, r)
    if ratio >= cfg.fuzzy_min_ratio:
        return True, f"fuzzy={ratio:.0f}"
    return False, f"not similar (fuzzy={ratio:.0f}, metaphone {jellyfish.metaphone(o)}!={jellyfish.metaphone(r)})"

def validate_edit(e: dict, seg_by_id: dict, glossary: list[str], cfg: RefineConfig, names: set | None = None) -> tuple[bool, str]:
    "Return (ok, reason). reason is 'ok' or why it was rejected."
    try:
        sid = int(e["segment_id"]); orig = str(e["original"]); rep = str(e["replacement"])
        conf = float(e.get("confidence", 1.0))
    except (KeyError, TypeError, ValueError):
        return False, "malformed"
    if sid not in seg_by_id:
        return False, "unknown segment"
    text = seg_by_id[sid]["text"]
    if not orig.strip() or not rep.strip():
        return False, "empty"
    if orig.strip() == rep.strip():
        return False, "no-op"
    if orig not in text:
        return False, "original not found verbatim"
    if conf < cfg.min_confidence:
        return False, f"confidence {conf:.2f} < {cfg.min_confidence}"
    ow, rw = tokens(orig), tokens(rep)
    if len(ow) > cfg.max_original_words:
        return False, f"too long ({len(ow)} words)"
    if protected_in(orig) != protected_in(rep):
        return False, f"touches protected tokens {protected_in(orig)} -> {protected_in(rep)}"
    # the protected tokens of the WHOLE segment must be unchanged after applying
    if protected_in(text) != protected_in(text.replace(orig, rep, 1)):
        return False, "would alter protected tokens in segment"
    gl = {g.lower() for g in glossary}
    name_pool = set(capitalised_names(text, glossary)) | (names if names is not None else global_names(list(seg_by_id.values()), glossary))
    touched = [t for t in ow if t in name_pool and t.lower() not in gl and t not in rw]
    if touched and rep.strip().lower() not in gl:
        return False, f"touches name(s) {touched}"
    ok, why = phonetically_plausible(orig, rep, glossary, cfg)
    if not ok:
        return False, why
    if why != "spelled-letters" and abs(len(ow) - len(rw)) > cfg.max_word_delta:
        return False, f"length change {len(ow)}->{len(rw)} words"
    return True, f"ok ({why})"

def validate_chunk_edits(edits: list, seg_by_id: dict, glossary: list[str], cfg: RefineConfig, seen: set, names: set | None = None):
    accepted, rejected = [], []
    for e in edits[: cfg.max_edits_per_chunk * 2]:
        key = (e.get("segment_id"), e.get("original"))
        if key in seen:
            continue                         # duplicate from overlapping chunk
        ok, why = validate_edit(e, seg_by_id, glossary, cfg, names)
        (accepted if ok else rejected).append({**e, "status": "accepted" if ok else "rejected", "check": why})
        seen.add(key)
        if len(accepted) >= cfg.max_edits_per_chunk:
            break
    return accepted, rejected

print("validators ready")

validators ready


## Cell 9 — Self-test the validators (no model needed)

Hand-written edits: four genuine fixes that must pass, six kinds of damage that must be rejected.
Run this after any change to Cell 5 or Cell 8.

In [ ]:
_test_segs = {1: {"id": 1, "text": "The post grass database crashed twice, Priya will fix it by Friday."},
              2: {"id": 2, "text": "We will not ship on the fifteenth, build time is fifty minutes."},
              3: {"id": 3, "text": "The see eye see dee pipeline pushes to rod via the A P I."},
              4: {"id": 4, "text": "Priya will send the terror form scripts."}}
_cases = [
    ({"segment_id": 1, "original": "post grass", "replacement": "PostgreSQL", "confidence": 0.9}, True),
    ({"segment_id": 3, "original": "see eye see dee", "replacement": "CI/CD", "confidence": 0.9}, True),
    ({"segment_id": 3, "original": "rod", "replacement": "prod", "confidence": 0.8}, True),
    ({"segment_id": 3, "original": "A P I", "replacement": "API", "confidence": 0.9}, True),
    ({"segment_id": 2, "original": "will not", "replacement": "will", "confidence": 0.9}, False),        # negation
    ({"segment_id": 2, "original": "fifty", "replacement": "fifteen", "confidence": 0.9}, False),        # number
    ({"segment_id": 1, "original": "Priya", "replacement": "Prya", "confidence": 0.9}, False),           # name
    ({"segment_id": 4, "original": "Priya", "replacement": "Pria", "confidence": 0.9}, False),           # sentence-initial name
    ({"segment_id": 4, "original": "terror form", "replacement": "Terraform", "confidence": 0.9}, True),
    ({"segment_id": 1, "original": "crashed twice", "replacement": "failed", "confidence": 0.9}, False), # rewrite
    ({"segment_id": 1, "original": "postgres", "replacement": "PostgreSQL", "confidence": 0.9}, False),  # not verbatim
    ({"segment_id": 1, "original": "post grass", "replacement": "PostgreSQL", "confidence": 0.3}, False),# low conf
]
fails = 0
for e, expect in _cases:
    ok, why = validate_edit(e, _test_segs, [], CFG)
    flag = "PASS" if ok == expect else "FAIL"; fails += ok != expect
    print(f"{flag}  {'accept' if ok else 'reject':<6} {e['original']!r:>18} -> {e['replacement']!r:<14} {why}")
print("\nALL GOOD" if not fails else f"\n{fails} validator test(s) failed — fix before running the model")

PASS  accept       'post grass' -> 'PostgreSQL'   ok (fuzzy=74)
PASS  accept  'see eye see dee' -> 'CI/CD'        ok (spelled-letters)
PASS  accept              'rod' -> 'prod'         ok (fuzzy=86)
PASS  accept            'A P I' -> 'API'          ok (spelled-letters)
PASS  reject         'will not' -> 'will'         touches protected tokens ['will', 'not'] -> ['will']
PASS  reject            'fifty' -> 'fifteen'      touches protected tokens ['fifty'] -> ['fifteen']
PASS  reject            'Priya' -> 'Prya'         touches name(s) ['Priya']
PASS  reject            'Priya' -> 'Pria'         touches name(s) ['Priya']
PASS  accept      'terror form' -> 'Terraform'    ok (fuzzy=84)
PASS  reject    'crashed twice' -> 'failed'       touches protected tokens ['twice'] -> []
PASS  reject         'postgres' -> 'PostgreSQL'   original not found verbatim
PASS  reject       'post grass' -> 'PostgreSQL'   confidence 0.30 < 0.6

ALL GOOD


## Cell 10 — Load the refiner LLM (Qwen3-8B, 4-bit)

~5.5 GB VRAM. First run downloads ~16 GB of safetensors (a few minutes). **Free the Whisper model first** if it's
still loaded in this runtime. Thinking mode is disabled so the output is pure JSON.

In [ ]:
import torch, time, gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

SAVE_4BIT_TO_DRIVE = True

def load_llm(cfg):
    t0 = time.time()
    local = MODELS_DIR / (cfg.model_id.replace("/", "__") + "-4bit")
    if (local / "config.json").exists():
        print("Loading 4-bit copy from Drive:", local)
        tok = AutoTokenizer.from_pretrained(local)
        model = AutoModelForCausalLM.from_pretrained(local, device_map="auto")
        source = "drive"
    else:
        print("Downloading from Hugging Face:", cfg.model_id, "(one-time)")
        tok = AutoTokenizer.from_pretrained(cfg.model_id)
        quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                   bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True) if cfg.load_in_4bit else None
        model = AutoModelForCausalLM.from_pretrained(cfg.model_id, quantization_config=quant,
                                                     device_map="auto", torch_dtype=torch.float16)
        source = "hub"
    model.eval()
    if source == "hub" and SAVE_4BIT_TO_DRIVE and cfg.load_in_4bit:
        print("Saving 4-bit copy to Drive (one-time, ~5.5 GB, a few minutes)...")
        model.save_pretrained(local, safe_serialization=True); tok.save_pretrained(local)
        print("Saved ->", local)
    mem = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
    print(f"Loaded {cfg.model_id} from {source} in {time.time()-t0:.0f}s | GPU mem {mem:.1f} GB")
    return tok, model

tokenizer, llm = load_llm(CFG)

def llm_generate(system: str, user: str, cfg: RefineConfig = CFG) -> str:
    "Greedy (temperature 0) generation; Qwen3 thinking disabled; returns the raw text."
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    kw = {"enable_thinking": False} if "qwen3" in cfg.model_id.lower() else {}
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, **kw)
    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=cfg.max_new_tokens, do_sample=False,
                           pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loaded Qwen/Qwen3-8B in 1112s | GPU mem 6.1 GB


## Cell 11 — Call + parse (retry once on bad JSON, then fall back to zero edits)

In [ ]:
def parse_edits(text: str) -> list | None:
    "Extract {'edits': [...]} from model output; tolerate code fences / leading prose. None if unparseable."
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()
    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M).strip()
    m = re.search(r"\{.*\}", text, flags=re.S)
    if not m:
        return [] if text.strip() in ("", "[]") else None
    try:
        obj = json.loads(m.group(0))
    except json.JSONDecodeError:
        return None
    edits = obj.get("edits", []) if isinstance(obj, dict) else obj
    return edits if isinstance(edits, list) else None

def refine_chunk(chunk: list[dict], glossary: list[str], suspect: dict, cfg: RefineConfig) -> tuple[list, list, dict]:
    user = build_user_message(chunk, glossary, suspect, cfg.meeting_context)
    raw = llm_generate(SYSTEM_PROMPT, user, cfg)
    edits = parse_edits(raw); attempts = 1
    if edits is None:
        raw = llm_generate(SYSTEM_PROMPT, user + '\n\nYour previous reply was not valid JSON. Reply with ONLY {"edits":[...]}.', cfg)
        edits = parse_edits(raw); attempts = 2
    warn = None
    if edits is None:
        edits, warn = [], "LLM returned invalid JSON twice; chunk left unedited"
    return edits, raw, {"attempts": attempts, "warning": warn}

print("refine_chunk ready")

refine_chunk ready


## Cell 12 — Run Stage 2 over all chunks

Produces the accepted/rejected edit log and the refined segments. Raw segments are never modified.

In [ ]:
import copy

def apply_edits(segments: list[dict], accepted: list[dict]) -> list[dict]:
    refined = copy.deepcopy(segments)
    by_id = {s["id"]: s for s in refined}
    for e in accepted:
        s = by_id[int(e["segment_id"])]
        if e["original"] in s["text"]:
            s["text"] = s["text"].replace(e["original"], e["replacement"], 1)
            e["applied"] = True
        else:
            e["applied"] = False; e["check"] = "superseded by an earlier edit in the same segment"
    return refined

def run_stage2(segments: list[dict], glossary: list[str], suspect: dict, cfg: RefineConfig, on_progress=None) -> dict:
    t0 = time.time(); seg_by_id = {s["id"]: s for s in segments}
    chunks = make_chunks(segments, cfg.chunk_segments, cfg.chunk_overlap)
    accepted, rejected, warnings, seen = [], [], [], set()
    for ci, chunk in enumerate(chunks):
        edits, raw, info = refine_chunk(chunk, glossary, suspect, cfg)
        if info["warning"]:
            warnings.append(f"chunk {ci+1}: {info['warning']}")
        a, r = validate_chunk_edits(edits, seg_by_id, glossary, cfg, seen, NAMES)
        accepted += a; rejected += r
        if on_progress:
            on_progress("refining", (ci + 1) / len(chunks))
    refined = apply_edits(segments, accepted)
    applied = [e for e in accepted if e.get("applied")]
    return {
        "refined_segments": refined,
        "refined_text": " ".join(s["text"] for s in refined),
        "edits_applied": applied,
        "edits_rejected": rejected + [e for e in accepted if not e.get("applied")],
        "warnings": warnings,
        "meta": {"refiner": f"{cfg.model_id} (4-bit NF4)" if cfg.load_in_4bit else cfg.model_id,
                 "n_chunks": len(chunks), "glossary": glossary, "n_suspect_segments": len(suspect),
                 "settings": {k: v for k, v in vars(cfg).items() if k != "glossary"},
                 "elapsed_s": round(time.time() - t0, 1)},
    }

def show_progress(stage, p): print(f"  [{p*100:5.1f}%] {stage}")

R2 = run_stage2(RAW_SEGMENTS, GLOSSARY, SUSPECT, CFG, on_progress=show_progress)
print(f"\n{R2['meta']['refiner']} | {R2['meta']['n_chunks']} chunk(s) in {R2['meta']['elapsed_s']}s | "
      f"applied {len(R2['edits_applied'])} | rejected {len(R2['edits_rejected'])} | warnings {len(R2['warnings'])}")

  [100.0%] refining

Qwen/Qwen3-8B (4-bit NF4) | 1 chunk(s) in 6.4s | applied 2 | rejected 0 | warnings 0


## Cell 13 — Inspect: edit log and side-by-side diff

This is what the UI's "raw vs refined with highlighted edits" view is built from.

In [ ]:
import difflib

print("=== APPLIED EDITS ===")
for e in R2["edits_applied"]:
    print(f"  seg {int(e['segment_id']):03d}  {e['original']!r} -> {e['replacement']!r}   [{e['check']}]  conf={e.get('confidence')}")
print("\n=== REJECTED (model proposed, code refused) ===")
for e in R2["edits_rejected"]:
    print(f"  seg {e.get('segment_id')!s:>3}  {e.get('original')!r} -> {e.get('replacement')!r}   REASON: {e['check']}")
for w in R2["warnings"]:
    print("WARNING:", w)

raw_text = " ".join(s["text"] for s in RAW_SEGMENTS)
sm = difflib.SequenceMatcher(None, raw_text.split(), R2["refined_text"].split())
changed = sum(max(i2 - i1, j2 - j1) for tag, i1, i2, j1, j2 in sm.get_opcodes() if tag != "equal")
print(f"\nedit ratio: {changed} / {len(raw_text.split())} words = {100*changed/max(1,len(raw_text.split())):.2f}%")

print("\n=== CHANGED SEGMENTS (raw | refined) ===")
for a, b in zip(RAW_SEGMENTS, R2["refined_segments"]):
    if a["text"] != b["text"]:
        print(f"[{a['id']:03d}] RAW : {a['text']}\n      REF : {b['text']}\n")

=== APPLIED EDITS ===
  seg 001  'pie torch' -> 'pytorch'   [ok (metaphone)]  conf=0.95
  seg 003  'atom optimizer' -> 'Adam optimizer'   [ok (metaphone)]  conf=0.85

=== REJECTED (model proposed, code refused) ===

edit ratio: 3 / 63 words = 4.76%

=== CHANGED SEGMENTS (raw | refined) ===
[001] RAW : We evaluated the pie torch model with low ra adaptations. The word error rate was zero point zero eight, not zero point eight.
      REF : We evaluated the pytorch model with low ra adaptations. The word error rate was zero point zero eight, not zero point eight.

[003] RAW : We also observed that the embedding vectors from bert did not converge when using the atom optimizer with weight decay. Let's switch to adam w instead.
      REF : We also observed that the embedding vectors from bert did not converge when using the Adam optimizer with weight decay. Let's switch to adam w instead.



## Cell 14 — Damage check (automatic)

Independent of the validators: compares the protected-token sequence and the name set of raw vs refined.
If anything differs, the validators have a hole — target is always 0.

In [ ]:
def damage_report(raw_segs: list, ref_segs: list, glossary: list[str]) -> dict:
    issues = []
    for a, b in zip(raw_segs, ref_segs):
        if protected_in(a["text"]) != protected_in(b["text"]):
            issues.append({"segment_id": a["id"], "type": "protected_tokens",
                           "raw": protected_in(a["text"]), "refined": protected_in(b["text"])})
        na, nb = set(capitalised_names(a["text"], glossary)), set(capitalised_names(b["text"], glossary))
        if na - nb:
            issues.append({"segment_id": a["id"], "type": "name_removed", "raw": sorted(na - nb)})
    return {"damage_count": len(issues), "issues": issues}

DMG = damage_report(RAW_SEGMENTS, R2["refined_segments"], GLOSSARY)
print("damage count:", DMG["damage_count"])
for i in DMG["issues"]:
    print("  ", i)

damage count: 0


## Cell 15 — Save Stage 2 artifacts

`refined_transcript.txt`, `refined_transcript_numbered.txt`, `edit_log.json`, and `stage2_record.json`
(the Stage 1 record with `refined_transcript` + `meta.models.refiner` filled in — this is what Stage 3 reads).

In [ ]:
from datetime import datetime

def fmt_ts(sec: float) -> str:
    sec = int(sec or 0); return f"{sec//60:02d}:{sec%60:02d}"

def numbered_lines(segments: list) -> str:
    return "\n".join(f"[L{s['id']:03d} {fmt_ts(s.get('start', 0))}] {s['text']}" for s in segments)

def save_stage2(s1: dict, r2: dict, dmg: dict, out_dir: Path) -> Path:
    run_dir = out_dir / f"stage2_{datetime.now():%Y%m%d_%H%M%S}"
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "refined_transcript.txt").write_text(r2["refined_text"], encoding="utf-8")
    (run_dir / "refined_transcript_numbered.txt").write_text(numbered_lines(r2["refined_segments"]), encoding="utf-8")
    edit_log = {
        "applied": r2["edits_applied"],
        "rejected": r2["edits_rejected"],
        "warnings": r2["warnings"],
        "damage_check": dmg,
        "meta": r2["meta"]
    }
    (run_dir / "edit_log.json").write_text(json.dumps(edit_log, indent=2, ensure_ascii=False), encoding="utf-8")

    record = copy.deepcopy(s1.get("record", {}))
    record.setdefault("meta", {}).setdefault("models", {})["refiner"] = r2["meta"]["refiner"]
    record["meta"]["refiner_settings"] = r2["meta"]["settings"]
    record["meta"]["refiner_warnings"] = r2["warnings"]
    record["asr2"] = s1.get("record", {}).get("asr2")  # Pass Parakeet findings through to Stage 3
    record["refined_transcript"] = {
        "text": r2["refined_text"],
        "segments": [{k: s.get(k) for k in ("id", "start", "end", "text")} for s in r2["refined_segments"]],
        "lines": numbered_lines(r2["refined_segments"]).split("\n"),
        "edits": [{k: e.get(k) for k in ("segment_id", "original", "replacement", "reason", "confidence", "check")}
                  for e in r2["edits_applied"]],
        "rejected_edits": len(r2["edits_rejected"]),
        "damage_count": dmg["damage_count"],
    }
    (run_dir / "stage2_record.json").write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Saved to", run_dir)
    return run_dir

RUN_DIR = save_stage2(S1, R2, DMG, OUT_DIR)

Saved to /content/meeting-assistant/outputs/stage2_20261007_124546
  edit_log.json                          0.9 KB
  refined_transcript.txt                 0.3 KB
  refined_transcript_numbered.txt        0.3 KB
  stage2_record.json                     8.6 KB


## Cell 16 — Evaluation on a clip with planted errors (tech-note table)

Fill `PLANTED` with the jargon you deliberately mispronounced/misrecognised in your eval clip and the correct
form. Reports term-fix rate, damage rate and edit ratio — the three Stage 2 numbers the playbook asks for.

In [ ]:
PLANTED = {           # "what Whisper produced" : "what it should be"   (leave empty to skip)
    # "post grass": "PostgreSQL",
    # "see eye see dee": "CI/CD",
    # "pie torch": "PyTorch",
}

if PLANTED:
    raw_l, ref_l = raw_text.lower(), R2["refined_text"].lower()
    rows = []
    for wrong, right in PLANTED.items():
        present_raw = wrong.lower() in raw_l
        fixed = right.lower() in ref_l and wrong.lower() not in ref_l
        rows.append((wrong, right, present_raw, fixed))
        print(f"  {wrong!r:>20} -> {right!r:<14} in raw: {present_raw!s:<5} fixed: {fixed}")
    n_planted = sum(r[2] for r in rows); n_fixed = sum(r[3] for r in rows if r[2])
    print(f"\nterm fix rate : {n_fixed}/{n_planted} = {100*n_fixed/max(1,n_planted):.0f}%")
    print(f"damage rate   : {DMG['damage_count']} (target 0)")
    print(f"edit ratio    : {100*changed/max(1,len(raw_text.split())):.2f}% of words changed")
    print(f"rejected edits: {len(R2['edits_rejected'])} (model proposals the validators refused)")
else:
    print("PLANTED is empty — skipping. (Still useful: damage rate", DMG["damage_count"], "and edit ratio above.)")

PLANTED is empty — skipping. (Still useful: damage rate 0 and edit ratio above.)


## Cell 17 — Free the GPU before Stage 3

Stage 3 loads a different model (Qwen2.5-7B-Instruct). Release this one first.

In [ ]:
def free_llm():
    global llm, tokenizer
    for name in ("llm", "tokenizer"):
        if name in globals():
            del globals()[name]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"GPU mem now {torch.cuda.memory_allocated()/1e9:.1f} GB")

# free_llm()   # uncomment when moving on to Stage 3 in this runtime